# 04 - Reproducibility Checklist

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain **why reproducibility matters** and the real-world consequences of non-reproducible results
- Identify the **sources of non-determinism** in deep learning (seeds, data loading, GPU ops, floating point)
- Apply a comprehensive **reproducibility checklist** to any PyTorch training script
- Use `src/utils/seed.py` to set all random seeds correctly
- Configure **deterministic algorithms** and **DataLoader worker seeds**
- Log **hardware and software environment** details for audit trails
- Implement a complete **reproducibility template** with `worker_init_fn`

## Prerequisites

- PyTorch fundamentals (training loops, DataLoaders)
- Understanding of random number generators and seeds
- Previous notebooks: [01 - Experiment Tracking](01_Experiment_Tracking_Lightweight.ipynb), [02 - Model Saving & Loading](02_Model_Saving_Loading_Inference_Pipeline.ipynb)

## Table of Contents

1. [Why Reproducibility Matters](#1-why-reproducibility-matters)
2. [Sources of Non-Determinism](#2-sources-of-non-determinism)
3. [The Reproducibility Checklist](#3-the-reproducibility-checklist)
4. [Setting Seeds with src/utils/seed.py](#4-setting-seeds-with-srcutilsseedpy)
5. [Demonstrating Non-Determinism and Fixes](#5-demonstrating-non-determinism-and-fixes)
6. [Complete Reproducibility Template](#6-complete-reproducibility-template)
7. [Hardware and Software Environment Logging](#7-hardware-and-software-environment-logging)
8. [Exercise: Make a Non-Reproducible Script Reproducible](#8-exercise-make-a-non-reproducible-script-reproducible)
9. [Common Mistakes & Debugging Tips](#9-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")

import os
import json
import platform
import random
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

# Deliberately NOT setting seed yet -- we will set it when needed
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## 1. Why Reproducibility Matters

Reproducibility means getting the **exact same results** when re-running the same code with the same data.

### Consequences of non-reproducibility

| Scenario | Impact |
|---|---|
| **Debugging** | Cannot tell if a code change improved results or if it was random variation |
| **Comparison** | Hyperparameter search is unreliable if results vary between identical runs |
| **Collaboration** | Teammates cannot reproduce your reported metrics |
| **Publication** | Reviewers and readers cannot verify claims |
| **Production** | Model behavior differs between training and retraining |
| **Regulation** | Finance, healthcare, and legal domains may require audit trails |

### Two levels of reproducibility

- **Exact reproducibility**: bit-for-bit identical results (same hardware, same software, deterministic mode)
- **Statistical reproducibility**: results are within expected variance (different hardware, different runs)

---
## 2. Sources of Non-Determinism

Deep learning training has **many** sources of randomness. Understanding each one is the first step to controlling them.

### 2.1 Random seeds

Multiple libraries each have their own random number generator (RNG):

| Library | RNG | Seed function |
|---|---|---|
| Python `random` | Mersenne Twister | `random.seed(s)` |
| NumPy | PCG64 (default) | `np.random.seed(s)` |
| PyTorch CPU | MT19937 | `torch.manual_seed(s)` |
| PyTorch CUDA | Philox | `torch.cuda.manual_seed_all(s)` |

**All four must be set** for full reproducibility.

### 2.2 Data loading order

- `DataLoader(shuffle=True)` uses its own RNG
- With `num_workers > 0`, each worker has its own RNG state
- Without a `worker_init_fn`, workers may produce identical random augmentations

### 2.3 GPU operations

- cuDNN auto-tuner (`torch.backends.cudnn.benchmark=True`) picks the fastest algorithm non-deterministically
- Some CUDA operations (atomicAdd, scatter, gather) are inherently non-deterministic
- `torch.use_deterministic_algorithms(True)` forces deterministic versions but may be slower

### 2.4 Floating point arithmetic

- Floating point addition is **not associative**: `(a + b) + c != a + (b + c)` in general
- Parallelism changes the order of operations, producing different rounding errors
- Different hardware (GPU models, CPU architectures) has different floating point behavior

In [ ]:
# Demonstrate floating point non-associativity
a = 1e-8
b = 1.0
c = -1.0

result1 = (a + b) + c
result2 = a + (b + c)

print(f"(a + b) + c = {result1}")
print(f"a + (b + c) = {result2}")
print(f"Difference:   {abs(result1 - result2)}")
print(f"Are they equal? {result1 == result2}")
print()

# This matters in reductions (sum over many elements)
torch.manual_seed(42)
vals = torch.randn(10000)

# Two different summation orders
sum_forward = vals.sum().item()
sum_reverse = vals.flip(0).sum().item()
print(f"Sum (forward): {sum_forward}")
print(f"Sum (reverse): {sum_reverse}")
print(f"Difference:    {abs(sum_forward - sum_reverse):.2e}")

---
## 3. The Reproducibility Checklist

Use this checklist for every training run:

- [ ] **Set all random seeds** (Python, NumPy, PyTorch CPU, PyTorch CUDA)
- [ ] **Enable deterministic algorithms** (`torch.use_deterministic_algorithms(True)` or `cudnn.deterministic`)
- [ ] **Disable cuDNN benchmark** (`torch.backends.cudnn.benchmark = False`)
- [ ] **Set DataLoader worker seeds** via `worker_init_fn` and `generator`
- [ ] **Pin software versions** (requirements.txt or environment.yml)
- [ ] **Log all hyperparameters** and configuration
- [ ] **Version your data** (hash, git-lfs, DVC, or at minimum record the source)
- [ ] **Save model checkpoints** with metadata (epoch, metrics, hyperparams)
- [ ] **Log environment** (OS, Python version, PyTorch version, GPU model)
- [ ] **Set `PYTHONHASHSEED`** environment variable
- [ ] **Use fixed train/val/test splits** (save indices or use seeded splitting)

---
## 4. Setting Seeds with src/utils/seed.py

Our project provides `set_global_seed()` in `src/utils/seed.py`. Let us examine what it does and verify it works.

In [ ]:
# Let's look at our seed utility
from src.utils.seed import set_global_seed
import inspect

print(inspect.getsource(set_global_seed))

In [ ]:
# Verify: calling set_global_seed produces identical results

def generate_random_values(seed: int) -> Dict[str, Any]:
    """Generate random values from all RNG sources after setting seed."""
    set_global_seed(seed)
    return {
        "python_random": random.random(),
        "numpy_random": np.random.rand(),
        "torch_cpu": torch.randn(1).item(),
        "torch_randint": torch.randint(0, 1000, (1,)).item(),
    }

# Run twice with the same seed
run1 = generate_random_values(42)
run2 = generate_random_values(42)

print(f"{'Source':<20} {'Run 1':>15} {'Run 2':>15} {'Match':>8}")
print("-" * 60)
for key in run1:
    match = run1[key] == run2[key]
    print(f"{key:<20} {run1[key]:>15.8f} {run2[key]:>15.8f} {str(match):>8}")

# Now run with a different seed
run3 = generate_random_values(123)
print(f"\nWith seed=123:")
for key in run1:
    print(f"  {key}: {run3[key]:.8f} (was {run1[key]:.8f})")

### What `set_global_seed` covers

- `random.seed(seed)` -- Python's built-in RNG
- `np.random.seed(seed)` -- NumPy's legacy RNG
- `torch.manual_seed(seed)` -- PyTorch CPU RNG
- `torch.cuda.manual_seed(seed)` / `manual_seed_all(seed)` -- PyTorch GPU RNG(s)
- `torch.backends.cudnn.deterministic = True` -- Force deterministic cuDNN
- `torch.backends.cudnn.benchmark = False` -- Disable auto-tuner
- `os.environ["PYTHONHASHSEED"] = str(seed)` -- Deterministic Python hashing

---
## 5. Demonstrating Non-Determinism and Fixes

Let us see non-determinism in action and then fix it step by step.

In [ ]:
# Create a small dataset
set_global_seed(42)
N = 500
X_data = torch.randn(N, 20)
y_data = (X_data @ torch.randn(20, 5)).argmax(dim=1)


class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 5),
        )

    def forward(self, x):
        return self.net(x)


print(f"Dataset: {N} samples, 20 features, 5 classes")

In [ ]:
# Demonstration: training WITHOUT setting seeds is non-reproducible

def train_without_seed(n_epochs: int = 10) -> List[float]:
    """Train without setting any seeds. Results will differ between calls."""
    # Deliberately NOT setting seed
    model = SmallMLP()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X_data, y_data), batch_size=32, shuffle=True)

    losses = []
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        n = 0
        for Xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n += 1
        losses.append(epoch_loss / n)
    return losses


# Run 3 times -- losses will differ
print("Training WITHOUT seeds (non-reproducible):")
print(f"{'Epoch':<8}", end="")
for run in range(1, 4):
    print(f"{'Run ' + str(run):>12}", end="")
print()
print("-" * 44)

all_runs_no_seed = [train_without_seed(10) for _ in range(3)]
for epoch in range(10):
    print(f"{epoch+1:<8}", end="")
    for run_losses in all_runs_no_seed:
        print(f"{run_losses[epoch]:>12.4f}", end="")
    print()

# Show final loss variance
final_losses = [r[-1] for r in all_runs_no_seed]
print(f"\nFinal loss range: {min(final_losses):.4f} - {max(final_losses):.4f} "
      f"(spread: {max(final_losses) - min(final_losses):.4f})")

In [ ]:
# Fix: training WITH seeds IS reproducible

def train_with_seed(seed: int = 42, n_epochs: int = 10) -> List[float]:
    """Train with proper seed setting. Results are identical between calls."""
    set_global_seed(seed)

    model = SmallMLP()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    # Use a seeded generator for the DataLoader
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(
        TensorDataset(X_data, y_data),
        batch_size=32, shuffle=True,
        generator=g,
    )

    losses = []
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        n = 0
        for Xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n += 1
        losses.append(epoch_loss / n)
    return losses


# Run 3 times -- losses will be identical
print("Training WITH seeds (reproducible):")
print(f"{'Epoch':<8}", end="")
for run in range(1, 4):
    print(f"{'Run ' + str(run):>12}", end="")
print()
print("-" * 44)

all_runs_with_seed = [train_with_seed(42, 10) for _ in range(3)]
for epoch in range(10):
    print(f"{epoch+1:<8}", end="")
    for run_losses in all_runs_with_seed:
        print(f"{run_losses[epoch]:>12.4f}", end="")
    print()

# Verify exact match
all_match = all(
    all_runs_with_seed[0][e] == all_runs_with_seed[i][e]
    for i in range(1, 3)
    for e in range(10)
)
print(f"\nAll runs exactly identical: {all_match}")

In [ ]:
# Visualize reproducibility vs non-reproducibility
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, 11)

for i, losses in enumerate(all_runs_no_seed):
    ax1.plot(epochs, losses, "o-", markersize=3, label=f"Run {i+1}")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss")
ax1.set_title("WITHOUT Seeds (Non-Reproducible)")
ax1.legend()
ax1.grid(True, alpha=0.3)

for i, losses in enumerate(all_runs_with_seed):
    style = ["o-", "s--", "^:"][i]
    ax2.plot(epochs, losses, style, markersize=4, label=f"Run {i+1}")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Train Loss")
ax2.set_title("WITH Seeds (Reproducible)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### DataLoader worker seeds

When using `num_workers > 0`, each worker process has its own RNG state. Without a `worker_init_fn`, workers may produce correlated random sequences (e.g., identical data augmentations).

The fix: provide a `worker_init_fn` that seeds each worker uniquely.

In [ ]:
def worker_init_fn(worker_id: int) -> None:
    """Seed each DataLoader worker for reproducibility.

    Each worker gets a unique seed derived from the base seed and worker_id.
    This ensures:
    1. Different workers produce different random sequences
    2. The sequences are reproducible across runs
    """
    # PyTorch sets each worker's seed to base_seed + worker_id.
    # We use that to seed numpy and random as well.
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# Example: creating a fully reproducible DataLoader
def make_reproducible_loader(
    dataset,
    batch_size: int = 32,
    shuffle: bool = True,
    seed: int = 42,
    num_workers: int = 0,
) -> DataLoader:
    """Create a DataLoader with reproducible shuffling and worker seeding."""
    g = torch.Generator()
    g.manual_seed(seed)

    kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "generator": g,
        "worker_init_fn": worker_init_fn,
        "num_workers": num_workers,
    }
    if num_workers > 0:
        kwargs["persistent_workers"] = True

    return DataLoader(dataset, **kwargs)


# Verify reproducibility
dataset = TensorDataset(X_data, y_data)

# First pass
loader1 = make_reproducible_loader(dataset, batch_size=32, seed=42)
batches1 = [yb.tolist() for _, yb in loader1]

# Second pass (new loader, same seed)
loader2 = make_reproducible_loader(dataset, batch_size=32, seed=42)
batches2 = [yb.tolist() for _, yb in loader2]

print(f"Number of batches: {len(batches1)}")
print(f"Batch orders identical: {batches1 == batches2}")
print(f"First batch (run 1): {batches1[0][:8]}...")
print(f"First batch (run 2): {batches2[0][:8]}...")

---
## 6. Complete Reproducibility Template

Here is a complete, copy-paste-ready template that incorporates every checklist item.

In [ ]:
def reproducible_training_run(
    seed: int = 42,
    lr: float = 0.01,
    batch_size: int = 32,
    n_epochs: int = 15,
    hidden_dim: int = 64,
) -> Dict[str, Any]:
    """Complete reproducible training template.

    Incorporates all checklist items:
    - Global seed setting
    - Deterministic algorithms
    - Seeded DataLoader with worker_init_fn
    - Environment logging
    - Hyperparameter logging
    """

    # ========== 1. Set all seeds ==========
    set_global_seed(seed)

    # ========== 2. Enable deterministic algorithms (optional, may be slower) ==========
    # Uncomment the line below for strictest determinism (may error on some ops):
    # torch.use_deterministic_algorithms(True)

    # ========== 3. Log configuration ==========
    config = {
        "seed": seed,
        "lr": lr,
        "batch_size": batch_size,
        "n_epochs": n_epochs,
        "hidden_dim": hidden_dim,
        "optimizer": "Adam",
        "loss_fn": "CrossEntropyLoss",
    }

    # ========== 4. Log environment ==========
    env_info = {
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "numpy_version": np.__version__,
        "os": f"{platform.system()} {platform.release()}",
        "cpu": platform.processor(),
        "cuda_available": torch.cuda.is_available(),
    }
    if torch.cuda.is_available():
        env_info["gpu"] = torch.cuda.get_device_name(0)
        env_info["cuda_version"] = torch.version.cuda

    # ========== 5. Prepare data with fixed split ==========
    # Use the global seed for data splitting
    split_idx = int(0.8 * len(X_data))
    perm = torch.randperm(len(X_data))  # Seeded by set_global_seed
    X_tr = X_data[perm[:split_idx]]
    y_tr = y_data[perm[:split_idx]]
    X_va = X_data[perm[split_idx:]]
    y_va = y_data[perm[split_idx:]]

    # ========== 6. Create reproducible DataLoaders ==========
    train_loader = make_reproducible_loader(
        TensorDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True,
        seed=seed,
    )
    val_loader = DataLoader(
        TensorDataset(X_va, y_va),
        batch_size=batch_size * 2,
        shuffle=False,  # No need to shuffle validation
    )

    # ========== 7. Build model ==========
    model = SmallMLP()  # Weights initialized from seeded RNG
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    # ========== 8. Train ==========
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, n_epochs + 1):
        # Train
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        history["train_loss"].append(epoch_loss / n_batches)

        # Validate
        model.eval()
        val_loss = 0.0
        correct = total = 0
        n_val = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                out = model(Xb)
                val_loss += loss_fn(out, yb).item()
                correct += (out.argmax(1) == yb).sum().item()
                total += yb.size(0)
                n_val += 1
        history["val_loss"].append(val_loss / n_val)
        history["val_acc"].append(correct / total)

    # ========== 9. Return everything for verification ==========
    return {
        "config": config,
        "environment": env_info,
        "history": history,
        "final_val_acc": history["val_acc"][-1],
        "final_train_loss": history["train_loss"][-1],
    }


# Run the template 3 times and verify reproducibility
results = []
for run in range(1, 4):
    result = reproducible_training_run(seed=42)
    results.append(result)
    print(f"Run {run}: final_val_acc={result['final_val_acc']:.6f}, "
          f"final_train_loss={result['final_train_loss']:.6f}")

# Check exact match
all_identical = all(
    results[0]["history"]["train_loss"][e] == results[i]["history"]["train_loss"][e]
    for i in range(1, 3)
    for e in range(15)
)
print(f"\nAll 3 runs produce identical training curves: {all_identical}")

In [ ]:
# Verify with a different seed produces different (but still reproducible) results
result_seed99_a = reproducible_training_run(seed=99)
result_seed99_b = reproducible_training_run(seed=99)

print(f"Seed=42: val_acc={results[0]['final_val_acc']:.6f}")
print(f"Seed=99 (run A): val_acc={result_seed99_a['final_val_acc']:.6f}")
print(f"Seed=99 (run B): val_acc={result_seed99_b['final_val_acc']:.6f}")
print(f"\nSeed=99 runs identical: "
      f"{result_seed99_a['history']['train_loss'] == result_seed99_b['history']['train_loss']}")
print(f"Seed=42 vs Seed=99 different: "
      f"{results[0]['history']['train_loss'] != result_seed99_a['history']['train_loss']}")

---
## 7. Hardware and Software Environment Logging

Even with perfect seeds, results may differ across hardware. Logging the environment lets you understand and explain differences.

In [ ]:
def log_environment() -> Dict[str, Any]:
    """Collect comprehensive hardware and software environment info."""
    env = {
        # Software
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "numpy_version": np.__version__,
        "os": platform.platform(),
        "timestamp": datetime.now().isoformat(),

        # CPU
        "cpu_arch": platform.machine(),
        "cpu_name": platform.processor(),
        "cpu_count": os.cpu_count(),

        # PyTorch config
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
    }

    # GPU info
    env["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        env["gpu_name"] = torch.cuda.get_device_name(0)
        env["gpu_count"] = torch.cuda.device_count()
        env["cuda_version"] = torch.version.cuda
        env["cudnn_version"] = str(torch.backends.cudnn.version())
        env["gpu_memory_gb"] = round(
            torch.cuda.get_device_properties(0).total_mem / 1e9, 2
        )

    # Installed package versions (selected)
    try:
        import sklearn
        env["sklearn_version"] = sklearn.__version__
    except ImportError:
        env["sklearn_version"] = "not installed"

    try:
        import matplotlib
        env["matplotlib_version"] = matplotlib.__version__
    except ImportError:
        env["matplotlib_version"] = "not installed"

    return env


# Log and display
env_info = log_environment()
print("Environment Report")
print("=" * 50)
for key, value in env_info.items():
    print(f"  {key:<25} {value}")

In [ ]:
def save_reproducibility_report(
    config: Dict[str, Any],
    environment: Dict[str, Any],
    path: str,
) -> None:
    """Save a JSON report containing everything needed to reproduce a run."""
    report = {
        "config": config,
        "environment": environment,
        "checklist": {
            "seeds_set": True,
            "cudnn_deterministic": environment.get("cudnn_deterministic", "N/A"),
            "cudnn_benchmark_disabled": not environment.get("cudnn_benchmark", True),
            "worker_init_fn_used": True,
            "data_split_seeded": True,
        },
        "timestamp": datetime.now().isoformat(),
    }

    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(report, f, indent=2, default=str)
    print(f"Reproducibility report saved to {path}")


# Save a report
save_reproducibility_report(
    config=results[0]["config"],
    environment=env_info,
    path="repro_report.json",
)

# Display it
with open("repro_report.json", "r") as f:
    report = json.load(f)
print("\nSaved report:")
print(json.dumps(report, indent=2))

# Clean up
os.remove("repro_report.json")

---
## 8. Exercise: Make a Non-Reproducible Script Reproducible

**Task:** The function below has **multiple reproducibility bugs**. Fix all of them.

### Bugs to find and fix
1. No global seed is set
2. DataLoader uses `shuffle=True` without a seeded `generator`
3. No `worker_init_fn`
4. Data split is not seeded
5. No environment or config logging

### Verification
After fixing, run the function 3 times and confirm all training losses are identical.

```python
def buggy_training(n_epochs: int = 10):
    """This function is NOT reproducible. Fix it!"""
    # Bug 1: no seed setting

    # Generate data
    X = torch.randn(500, 20)
    y = torch.randint(0, 5, (500,))

    # Bug 2/3/4: no seeded generator, no worker_init_fn, random split
    indices = torch.randperm(500)
    X_tr, y_tr = X[indices[:400]], y[indices[:400]]
    X_va, y_va = X[indices[400:]], y[indices[400:]]

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

    model = SmallMLP()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    losses = []
    for epoch in range(n_epochs):
        model.train()
        total = 0.0
        n = 0
        for Xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item()
            n += 1
        losses.append(total / n)

    # Bug 5: no logging
    return losses
```

In [ ]:
# Try the exercise yourself before looking at the solution below!







### Solution

In [ ]:
# ----- Solution -----

def fixed_training(seed: int = 42, n_epochs: int = 10) -> Dict[str, Any]:
    """Fixed version: fully reproducible training."""

    # Fix 1: Set all random seeds
    set_global_seed(seed)

    # Fix 5 (part 1): Log config
    config = {
        "seed": seed,
        "n_epochs": n_epochs,
        "lr": 0.01,
        "batch_size": 32,
    }

    # Generate data (now seeded)
    X = torch.randn(500, 20)
    y = torch.randint(0, 5, (500,))

    # Fix 4: Data split uses the seeded RNG
    indices = torch.randperm(500)  # Seeded by set_global_seed
    X_tr, y_tr = X[indices[:400]], y[indices[:400]]
    X_va, y_va = X[indices[400:]], y[indices[400:]]

    # Fix 2 & 3: Seeded generator + worker_init_fn
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=32,
        shuffle=True,
        generator=g,
        worker_init_fn=worker_init_fn,
    )

    model = SmallMLP()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    losses = []
    for epoch in range(n_epochs):
        model.train()
        total = 0.0
        n = 0
        for Xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item()
            n += 1
        losses.append(total / n)

    # Fix 5 (part 2): Log environment
    environment = log_environment()

    return {
        "losses": losses,
        "config": config,
        "environment": environment,
    }


# Verify: run 3 times
fixed_runs = [fixed_training(seed=42) for _ in range(3)]

print(f"{'Epoch':<8}", end="")
for run in range(1, 4):
    print(f"{'Run ' + str(run):>12}", end="")
print()
print("-" * 44)

for epoch in range(10):
    print(f"{epoch+1:<8}", end="")
    for run in fixed_runs:
        print(f"{run['losses'][epoch]:>12.4f}", end="")
    print()

# Verify exact match
all_identical = all(
    fixed_runs[0]["losses"][e] == fixed_runs[i]["losses"][e]
    for i in range(1, 3)
    for e in range(10)
)
print(f"\nAll runs identical: {all_identical}")

# Show logged info
print(f"\nConfig: {fixed_runs[0]['config']}")
print(f"Environment (selected):")
for key in ["python_version", "pytorch_version", "os"]:
    print(f"  {key}: {fixed_runs[0]['environment'][key]}")

---
## 9. Common Mistakes & Debugging Tips

**1. Setting seeds once but not before each run**
- Seeds get consumed as training progresses. To reproduce run N, you must reset all seeds before run N.
- Call `set_global_seed(seed)` at the very start of each training function.

**2. Forgetting DataLoader's own RNG**
- `DataLoader(shuffle=True)` uses a separate RNG for shuffling.
- Pass a `torch.Generator` seeded with your seed: `DataLoader(..., generator=g)`.

**3. Not seeding DataLoader workers**
- Without `worker_init_fn`, workers with `num_workers > 0` may produce identical or unpredictable random sequences.
- Always pass `worker_init_fn` that seeds `numpy` and `random` from `torch.initial_seed()`.

**4. Using `torch.backends.cudnn.benchmark = True`**
- The auto-tuner selects different algorithms non-deterministically.
- Set `benchmark = False` for reproducibility (slight speed cost).

**5. Expecting exact reproducibility across different GPUs**
- Different GPU architectures have different floating point behavior.
- Cross-GPU reproducibility is generally impossible at the bit level.
- Log the GPU model and accept statistical (not exact) reproducibility across hardware.

**6. Forgetting `PYTHONHASHSEED`**
- Python's `hash()` function is randomized by default (since Python 3.3).
- This affects dict iteration order and set operations.
- Set `os.environ["PYTHONHASHSEED"] = str(seed)` before any imports (our `set_global_seed` handles this).

**7. Not pinning package versions**
- A PyTorch minor version update can change default algorithm selections.
- Use `pip freeze > requirements.txt` or `conda env export > environment.yml`.

**8. Assuming determinism is free**
- `torch.use_deterministic_algorithms(True)` forces slower, deterministic CUDA kernels.
- Some operations have no deterministic implementation and will raise errors.
- The tradeoff: perfect reproducibility vs training speed.

**9. Seeding after model creation**
- Model weight initialization uses the RNG. If you set the seed after `model = MyModel()`, weights are random.
- Always set seeds **before** creating the model, optimizer, and data splits.

---

### Quick Reference: Reproducibility Checklist

```python
import os, random
import numpy as np
import torch

SEED = 42

# 1. Set all seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# 2. Deterministic backends
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# torch.use_deterministic_algorithms(True)  # strictest

# 3. Seeded DataLoader
g = torch.Generator().manual_seed(SEED)
loader = DataLoader(
    dataset, batch_size=32, shuffle=True,
    generator=g, worker_init_fn=worker_init_fn,
)

# 4. Log everything
# - config (hyperparams, seed)
# - environment (python, pytorch, numpy, os, gpu)
# - data version/hash
# - package versions (requirements.txt)
```

---

This is the final notebook in the DL800 (Deep Learning Operations / MLOps Lite) module. You now have the tools to track experiments, save and load models, profile performance, and ensure reproducibility.